# No Finding Quick Submission

Generate a fast baseline submission by copying the competition submission template and predicting `No Finding` for every test image.

## Configure Paths

The notebook searches common local, Colab, and Kaggle paths for `test_submission.csv`. Adjust `OUTPUT_PATH` if you want the CSV somewhere else.

In [ ]:
from pathlib import Path

import pandas as pd

ID_COLUMN = "filename"
LABEL_COLUMNS = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Enlarged Cardiomediastinum",
    "Fracture",
    "Lung Lesion",
    "Lung Opacity",
    "No Finding",
    "Pleural Effusion",
    "Pleural Other",
    "Pneumonia",
    "Pneumothorax",
]
EXPECTED_COLUMNS = [ID_COLUMN] + LABEL_COLUMNS

cwd = Path.cwd()
search_roots = [cwd, *cwd.parents]
template_candidates = []
for root in search_roots:
    template_candidates.extend(
        [
            root / "data" / "individual-test-chest-disease-detection" / "test_submission.csv",
            root / "data" / "chest-disease-detection" / "test_submission.csv",
            root / "test_submission.csv",
        ]
    )
template_candidates.extend(
    [
        Path("/content/input/chest-disease-detection/test_submission.csv"),
        Path("/content/input/individual-test-chest-disease-detection/test_submission.csv"),
        Path("/kaggle/input/chest-disease-detection/test_submission.csv"),
        Path("/kaggle/input/individual-test-chest-disease-detection/test_submission.csv"),
    ]
)

TEMPLATE_PATH = next((path for path in template_candidates if path.exists()), None)
if TEMPLATE_PATH is None:
    checked = "\n".join(str(path) for path in template_candidates)
    raise FileNotFoundError(f"Could not find test_submission.csv. Checked:\n{checked}")

project_root = next((root for root in search_roots if (root / "pyproject.toml").exists()), cwd)
OUTPUT_PATH = project_root / "outputs" / "submissions" / "submission_no_finding.csv"

TEMPLATE_PATH, OUTPUT_PATH


## Load Template

The competition expects the output CSV to preserve the same columns and filename order as `test_submission.csv`.

In [ ]:
template = pd.read_csv(TEMPLATE_PATH)

if list(template.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        "Template columns do not match the expected schema. "
        f"Found: {list(template.columns)}"
    )

template.head()


## Build Submission

Every row gets `No Finding = 1`; all other labels are set to `0`.

In [ ]:
submission = template.copy()
submission[LABEL_COLUMNS] = 0
submission["No Finding"] = 1

assert list(submission.columns) == list(template.columns)
assert submission[ID_COLUMN].astype(str).tolist() == template[ID_COLUMN].astype(str).tolist()
assert not submission[LABEL_COLUMNS].isna().any().any()
assert submission[LABEL_COLUMNS].isin([0, 1]).all().all()
assert submission["No Finding"].eq(1).all()
assert submission[[label for label in LABEL_COLUMNS if label != "No Finding"]].eq(0).all().all()

submission.head()


## Save CSV

The saved file is ready to upload as a quick baseline submission.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

print(f"Wrote {len(submission):,} rows to {OUTPUT_PATH}")
submission["No Finding"].value_counts().rename("count")
